# 15-bob. Etika, maxfiylik va huquqiy asoslar
## Kengaytirilgan amaliy ish: spam filtrining adolati

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/15-bob/15-3_spam-filtrining-adolati.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 15-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dasturda spam filtri asosan ingliz tilidagi xatlarda o'qitiladi, o'zbek tilidagi xatlar o'quv tanlanmasining atigi 2 foizini tashkil etadi. Filtr lug'ati ingliz tiliga moslashgani uchun o'zbekcha oddiy xatlarda ham "notanish so'zlar ulushi" yuqori bo'ladi. Spam xatlar ham notanish so'zlarga boy, chunki ular filtrni aldash uchun so'zlarni buzib yozadi. Model har ikki til uchun alohida baholanadi, so'ngra noxolislikni kamaytirishning ikki usuli solishtiriladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(15)
def xatlar(n, til, spam_ulushi=0.3):
    """Har bir xat 4 ta xususiyat bilan: notanish so'zlar ulushi, havolalar soni, shoshiltiruvchi so'zlar, jo'natuvchi yoshi."""
    y = (rng.random(n) < spam_ulushi).astype(int)
    notanish = np.where(y == 1, rng.normal(0.45, 0.12, n), rng.normal(0.10, 0.06, n))
    if til == "uz":                    # lug'at asosan ingliz tilida tuzilgan: o'zbekcha oddiy xatda ham notanish so'zlar ko'p
        notanish = np.where(y == 1, rng.normal(0.55, 0.12, n), rng.normal(0.38, 0.10, n))
    havola = rng.poisson(np.where(y == 1, 3.0, 1.0))
    shoshilinch = rng.poisson(np.where(y == 1, 2.0, 0.4))
    yangi_domen = (rng.random(n) < np.where(y == 1, 0.6, 0.1)).astype(float)
    return np.c_[notanish, havola, shoshilinch, yangi_domen], y

def baholash(model, chegara, sinov):
    natija = {}
    for til in ("en", "uz"):
        X, y = sinov[til]
        p = model.predict_proba(X)[:, 1] >= chegara[til]
        natija[til] = (p[y == 0].mean(), p[y == 1].mean())     # FPR va aniqlash darajasi
    return natija

sinov = {til: xatlar(4000, til) for til in ("en", "uz")}
def chop(nom, r):
    print(f"{nom:<46} en: FPR {r['en'][0]:.3f}, aniqlash {r['en'][1]:.2f} | "
          f"uz: FPR {r['uz'][0]:.3f}, aniqlash {r['uz'][1]:.2f}")

In [2]:
# 1. Asosan ingliz tilidagi xatlarda o'qitilgan filtr (o'zbekcha xatlar 2%)
Xe, ye = xatlar(9800, "en"); Xu, yu = xatlar(200, "uz")
m1 = LogisticRegression(max_iter=1000).fit(np.r_[Xe, Xu], np.r_[ye, yu])
chop("1) Asl model, umumiy chegara 0.5", baholash(m1, {"en": 0.5, "uz": 0.5}, sinov))

1) Asl model, umumiy chegara 0.5               en: FPR 0.002, aniqlash 0.95 | uz: FPR 0.378, aniqlash 0.99


In [3]:
# 2. Guruhlar uchun alohida chegara: o'zbekcha xatlarda FPR ingliz tilidagidek bo'lsin
Xv, yv = xatlar(2000, "uz")                                       # alohida tekshirish to'plami
p_v = m1.predict_proba(Xv)[:, 1][yv == 0]
fpr_en = baholash(m1, {"en": 0.5, "uz": 0.5}, sinov)["en"][0]
chop("2) Asl model, o'zbekcha uchun alohida chegara", baholash(m1, {"en": 0.5, "uz": np.quantile(p_v, 1 - fpr_en)}, sinov))

2) Asl model, o'zbekcha uchun alohida chegara  en: FPR 0.002, aniqlash 0.95 | uz: FPR 0.001, aniqlash 0.60


In [4]:
# 3. O'zbekcha xatlar ko'paytirilib, til xususiyati qo'shilgan model
def tilli(X, til): return np.c_[X, np.full(len(X), til == "uz"), X[:, 0] * (til == "uz")]
Xu2, yu2 = xatlar(3000, "uz")
m3 = LogisticRegression(max_iter=1000).fit(np.r_[tilli(Xe, "en"), tilli(Xu2, "uz")], np.r_[ye, yu2])
sinov3 = {til: (tilli(X, til), y) for til, (X, y) in sinov.items()}
chop("3) Ko'proq o'zbekcha ma'lumot + til xususiyati", baholash(m3, {"en": 0.5, "uz": 0.5}, sinov3))

3) Ko'proq o'zbekcha ma'lumot + til xususiyati en: FPR 0.003, aniqlash 0.97 | uz: FPR 0.028, aniqlash 0.86


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
1) Asl model, umumiy chegara 0.5               en: FPR 0.002, aniqlash 0.95 | uz: FPR 0.378, aniqlash 0.99
2) Asl model, o'zbekcha uchun alohida chegara  en: FPR 0.002, aniqlash 0.95 | uz: FPR 0.001, aniqlash 0.60
3) Ko'proq o'zbekcha ma'lumot + til xususiyati    en: FPR 0.003, aniqlash 0.97 | uz: FPR 0.028, aniqlash 0.86
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 15-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.